In [ ]:
import pandas as pd
from scipy.spatial import cKDTree
import numpy as np
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import learning_curve

In [ ]:
# Load data
data = pd.read_csv(r"mapped_s1_data.dat")

# Data separation
train_data = data[(data['YXML50'].isin([0, 1, 2]))].copy()  # Data with known labels

# Extract coordinates and features
original_data = train_data[train_data["T"] == 1]
coordinates = original_data[["X", "Y", "Z"]].values

kd_tree = cKDTree(coordinates)  # Build spatial index tree using cKDTree to accelerate nearest neighbor search
N = 728  # Number of nearest neighbors
# Query N nearest neighbors for each point (excluding self)
neighbors_indices = [kd_tree.query(point, k=N + 1)[1][1:] for point in coordinates]

scaler = StandardScaler()
data_features_scaled = scaler.fit_transform(original_data[["den", "sus", "res"]].values)
labels = original_data["YXML50"].values
ZK = original_data["T"].values

In [ ]:
# Data split
mask = (train_data['T'] == 1) 
original_indices = np.where(mask)[0]  # Get indices satisfying the condition

train_indices, test_indices = train_test_split(
    original_indices, test_size=0.2, random_state=42, stratify=labels[original_indices]
)

def create_spatial_features(neighbors_indices, features):

    n_samples = len(neighbors_indices)
    n_features = features.shape[1]
    # Initialize feature matrix
    X_spatial = np.zeros((n_samples, N * n_features))

    # Fill nearest neighbor features
    for i in range(n_samples):
        neighbor_features = features[neighbors_indices[i]]
        X_spatial[i] = neighbor_features.flatten()

    return X_spatial

In [ ]:
# Create base features and spatial features
X_base = data_features_scaled[original_indices]  # Original features
X_spatial = create_spatial_features(neighbors_indices, data_features_scaled)
X = np.hstack([X_base, X_spatial])  # Concatenate features

# Split training and test sets (stratified sampling already applied)
X_train = X[train_indices]
X_test = X[test_indices]
y_train = labels[train_indices]
y_test = labels[test_indices]

In [ ]:
# ===== Automatically compute class weights =====
from sklearn.utils.class_weight import compute_class_weight

# Compute class weights
classes = np.unique(y_train)
class_weights = compute_class_weight('balanced', classes=classes, y=y_train)
# Convert to dictionary format
class_weight_dict = dict(zip(classes, class_weights))

print(f"Automatically computed class weights: {class_weight_dict}")

# Create more refined weight parameter options
# Based on automatically computed weights, create several variants
weight_variants = [
    None,  # No imbalance handling
    'balanced',  # Use sklearn's balanced weights
    class_weight_dict,  # Use our computed balanced weights
]

# Parameter grid - add more class weight options
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],  # Add min_samples_leaf parameter to prevent overfitting
    'class_weight': weight_variants  # Use the various weight options we created
}

In [ ]:
# Create model
rf = RandomForestClassifier(n_jobs=-1, random_state=42, oob_score=False)

# Use stratified K-fold cross-validation to ensure consistent class proportions in each fold
from sklearn.model_selection import StratifiedKFold
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Grid search
grid_search = GridSearchCV(
    estimator=rf, 
    param_grid=param_grid, 
    cv=cv_strategy,  # Use stratified K-fold
    scoring='f1_macro',  # Use macro-averaged F1 as the primary evaluation metric
    n_jobs=-1,  # Use all available CPU cores
    verbose=1  # Show progress
)

# Train model
grid_search.fit(X_train, y_train)

# Best model
best_rf = grid_search.best_estimator_

# Output best parameters
print("\nBest parameters:", grid_search.best_params_)
print("Best cross-validation score (macro F1):", grid_search.best_score_)

In [ ]:
# Model evaluation
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

y_pred = best_rf.predict(X_test)

# Compute various metrics
print("\n=== Test set detailed evaluation metrics ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")

# Detailed classification report
print("\nClassification report:")
print(classification_report(y_test, y_pred, target_names=[f'Class {i}' for i in classes]))

In [ ]:
# Confusion matrix
conf_matrix = confusion_matrix(y_test, y_pred)
print("\nConfusion matrix:")
print(conf_matrix)

# Add probability prediction output
y_proba = best_rf.predict_proba(X_test)
print("\nProbability predictions for the first 5 samples:")
print(y_proba[:5])

In [ ]:
# Create evaluation metrics dictionary
from sklearn.metrics import precision_score, recall_score, f1_score

metrics = {
    "accuracy": accuracy_score(y_test, y_pred),
    "precision_macro": precision_score(y_test, y_pred, average='macro'),
    "recall_macro": recall_score(y_test, y_pred, average='macro'),
    "f1_macro": f1_score(y_test, y_pred, average='macro')
}

print("\nEvaluation metrics dictionary:")
for key, value in metrics.items():
    print(f"{key}: {value:.4f}")

print("Best Parameters:", grid_search.best_params_)
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print(conf_matrix)

In [ ]:
# Feature importance analysis (moved after best_rf is defined)
importances = best_rf.feature_importances_
std = np.std([tree.feature_importances_ for tree in best_rf.estimators_], axis=0)
indices = np.argsort(importances)[::-1]
'''
# Print feature importance
print("Feature ranking:")
for f in range(X.shape[1]):
    print(f"{f + 1}. Feature {indices[f]} ({importances[indices[f]]:.4f}) ± {std[indices[f]]:.4f}")
'''
# Visualize feature importance
plt.figure(figsize=(12, 8))
plt.title("Feature Importance")
plt.bar(range(X.shape[1]), importances[indices],color="r", yerr=std[indices], align="center")
plt.xticks(range(X.shape[1]), indices)
plt.xlim([-1, X.shape[1]])
plt.show()

In [ ]:
# Learning curve analysis (moved after best_rf is defined)
train_sizes, train_scores, test_scores = learning_curve(
    best_rf, X_train, y_train, cv=5,
    scoring='f1_macro', train_sizes=np.linspace(0.1, 1.0, 5)
)

train_mean = np.mean(train_scores, axis=1)
train_std = np.std(train_scores, axis=1)
test_mean = np.mean(test_scores, axis=1)
test_std = np.std(test_scores, axis=1)

plt.figure(figsize=(8, 6))
plt.plot(train_sizes, train_mean, 'o-', color='r', label='Training score')
plt.plot(train_sizes, test_mean, 'o-', color='g', label='Cross-validation score')
plt.fill_between(train_sizes, train_mean - train_std,train_mean + train_std, alpha=0.1, color='r')
plt.fill_between(train_sizes, test_mean - test_std,test_mean + test_std, alpha=0.1, color='g')
plt.xlabel('Training examples')
plt.ylabel('F1 Score (Macro)')
plt.legend(loc='best')
plt.show()

def plot_confusion_matrix(conf_matrix, class_names, normalize=False, title='Confusion Matrix'):
    plt.figure(figsize=(10, 8))
    plt.imshow(conf_matrix, interpolation='nearest', cmap=plt.cm.Blues)
    plt.title(title)
    plt.colorbar()
    # Set axis labels
    tick_marks = np.arange(len(class_names))
    plt.xticks(tick_marks, class_names, rotation=45)
    plt.yticks(tick_marks, class_names)
    # Add numeric labels
    thresh = conf_matrix.max() / 2.
    for i in range(conf_matrix.shape[0]):
        for j in range(conf_matrix.shape[1]):
            plt.text(j, i, format(conf_matrix[i, j], 'd'),
                     horizontalalignment="center",
                     color="white" if conf_matrix[i, j] > thresh else "black")

    plt.ylabel('True label')
    plt.xlabel('Predicted label')
    plt.tight_layout()
    plt.show()


class_names = ['Class 0', 'Class 1', 'Class 2']
plot_confusion_matrix(conf_matrix, class_names, normalize=True)
save_path = "autodl-tmp/Confusion Matrix"
plt.savefig(save_path, dpi=300, bbox_inches='tight')
print(f"Image saved to: {save_path}")